# Analytical null integration — summary

**Task** `analytical-null-md`, gate 2. This notebook is the review artifact
for the pull request: read this, not the branch diff, first.

## What this task did

The web query tool's null (`query_metapath_z` in `src/multi_dwpc_query.py`)
is replaced. The old null scored a user's gene set against `b = 20` random
same-size gene subsets drawn from the whole gene universe — Monte Carlo,
degree-blind, reseeded per call. The new null is the exact moments of the
same resampling scheme, computed in closed form by
`hetnetex_md.exact_resampling_moments` over a **capacity-stratified**
partition of the gene universe, via the adapter `analytical_gene_set_z`
(`src/analytical_null.py`). No `b`, no seed, no sampling error.

## The hypothesis

Two claims, both tested by the evidence below:

1. The Monte-Carlo null and the analytical null score genuinely different
   things, because the Monte-Carlo null is blind to gene degree and the
   analytical null is not — the reordering it produces should be visible in
   the figures below, not just claimed.
2. Replacing the null changes the *shape* of the cost (no more `b` to tune,
   deterministic output) without necessarily changing the query's practical
   cost, because end-to-end latency is dominated by disk I/O, not by
   null computation.

## Why stratify

The null must answer: is this gene set's connectivity to *this* target
higher than a random gene set's would be? That depends on what counts as "a
random gene set." A gene's raw DWPC to a target is confounded by its overall
connectivity (degree) in the network — high-degree genes score higher
against every target, enrichment or not. Stratifying removes that confound:
each of the user's genes is compared only against genes of similar
**capacity** (its total raw DWPC to every *other* target of the metapath,
excluding the tested target), so the resulting z-score reports whether these
genes reach this target more than genes with the same overall reach — not
more than an unstratified random draw would.

## What to expect below

- **Adapt-step evidence** — a per-metapath comparison table and two figures
  showing the analytical z sitting generally at or below the Monte-Carlo z,
  and the Monte-Carlo seed-to-seed spread the analytical null does not have.
- **Verify-step evidence** — timing tables and a figure showing the
  *measured*, not assumed, cost of the swap; a rank-agreement table and
  figure showing how much the honest null reorders the ranking.
- **Behaviour changes** — two changes a reviewer must accept, stated as
  design commitments.
- **Conclusions** — pointers to the audit, the full verification log, and
  the validation evidence this null is built on.


## Adapt-step evidence: the query backend

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

# Robust to being run either from this notebook's own directory (the
# intended way — see plan.md/design.md) or from the repository root.
BASE = Path.cwd()
if not (BASE / "tables").exists():
    BASE = Path("docs/tasks/analytical-null-md")
if not (BASE / "tables").exists():
    raise FileNotFoundError(
        "Could not locate tables/ and figures/. Run this notebook from "
        "docs/tasks/analytical-null-md/, its own directory."
    )

TABLES = BASE / "tables"
FIGURES = BASE / "figures"
print(f"Reading tables from:  {TABLES.resolve()}")
print(f"Reading figures from: {FIGURES.resolve()}")


In [ ]:
comparison = pd.read_csv(TABLES / "per_metapath_comparison.csv")
display(comparison.head())

n_total = len(comparison)
n_nan_analytical = comparison["z_analytical"].isna().sum()
n_nan_any_mc_seed = (
    comparison[["z_mc_seed42", "z_mc_seed43", "z_mc_seed44"]].isna().any(axis=1).sum()
)

print(f"Metapaths scored: {n_total}")
print(f"Analytical z is NaN (zero-variance null): {n_nan_analytical} / {n_total}")
print(f"NaN in at least one MC seed:              {n_nan_any_mc_seed} / {n_total}")


In [ ]:
Image(filename=str(FIGURES / "old_vs_new_z_scatter.png"))


**Figure 1 — old vs. new z, per metapath.** The Monte-Carlo z at each of
three seeds (42, 43, 44) plotted against the single analytical z, with
`y = x` for reference. The MC seeds sit systematically above the `y = x`
line for the highest-z metapaths — the naive, degree-blind null over-states
enrichment there relative to the degree-stratified analytical null — and
scatter on both sides of it near `z = 0`. Read plainly: the analytical z
sits generally at or below the Monte-Carlo z.


In [ ]:
Image(filename=str(FIGURES / "mc_seed_spread.png"))


**Figure 2 — Monte-Carlo seed spread vs. the analytical value.**
Per-metapath MC seed range (min-max across seeds 42/43/44, horizontal bars)
against the single deterministic analytical z (dot) — the determinism claim
made visible. Several metapaths (`GpPWpGpBP`, `GpBPpGpBP`, `GpMFpGpBP`) show
MC seed ranges spanning roughly 5-20 z units (min-max: 19.34, 15.88, 4.75
respectively) at `b = 20`; the analytical value has none, by construction —
the same query returns the same z every time.


## Verify-step evidence: timing

In [ ]:
timing = pd.read_csv(TABLES / "timing.csv")
warm_timing = pd.read_csv(TABLES / "warm_metapath_timing.csv")

display(timing)
display(warm_timing)

print("End-to-end median wall-clock (s), 3 runs each:")
display(timing.groupby("implementation")["wall_clock_s"].median())

print("Warm-matrix median wall-clock (s), 5 runs each:")
display(warm_timing.groupby(["metapath", "implementation"])["wall_clock_s"].median())


In [ ]:
Image(filename=str(FIGURES / "timing_comparison.png"))


**Figure 3 — timing comparison.** Two panels, one per table above. The
measured truth, plainly, per the corrected design (`decisions.md`,
2026-09-01):

- **End-to-end** (52 metapaths, disk-bound): new (analytical) 103 s vs old
  (Monte-Carlo, `b = 20`) 85 s — median of 3 runs each. Roughly unchanged;
  both implementations spend most of their wall-clock loading DWPC matrices
  from disk, not computing the null.
- **Warm-matrix** (matrix already resident, isolates the null-computation
  cost alone): the null step is slower under the analytical implementation —
  70 ms vs 8.6 ms on `GiGiGpBP` (median of 5 runs), 12.6 ms vs 7.5 ms on the
  smaller `GpBP`.
- **Why `b = 20` was a weak yardstick.** At the app's default `b = 20`, the
  old null was fast because it was imprecise: a null standard deviation
  estimated from only 20 draws carries roughly 16% relative error, which
  every z-score it feeds inherits. Twenty draws is cheap; twenty draws is
  also not a precise estimate.
- **The validated speedup is real, at a different scale.** The analytical
  moments price out at the Monte-Carlo sample size a comparable-precision
  null would need — validated at 214x against `B = 1,000` draws and 2,145x
  against `B = 10,000` draws (the capacity-hurdle-adaptive validation task,
  see Conclusions). At the app's much smaller `b = 20`, that crossover has
  not been reached for the metapaths measured here.
- **Correction on record.** The design's original expectation — an
  order-of-magnitude latency drop — was written against the wrong yardstick
  (`b = 20` vs the validated `B >= 1,000`) and is corrected in
  `decisions.md` (2026-09-01) rather than left standing against the
  measurement.


## Verify-step evidence: rank agreement

In [ ]:
rank_agreement = pd.read_csv(TABLES / "rank_agreement.csv")
display(rank_agreement.head())
print(f"Rows with both an analytical and an MC-seed-42 rank (n): {len(rank_agreement)}")


In [ ]:
Image(filename=str(FIGURES / "rank_agreement.png"))


**Figure 4 — rank agreement.** Analytical rank vs. Monte-Carlo (seed 42)
rank for the `n = 44` metapaths where both are defined (52 total minus 8
where either side is a zero-variance NaN). Reported in `verification.md`:
Spearman rho = 0.2605 (p = 0.0877) — not significant at alpha = 0.05, weak
agreement.

Read honestly, this is the expected and desired finding, not a defect: a
null that is blind to degree (Monte-Carlo, whole-gene-universe) and a null
that is not (analytical, capacity-stratified) should reorder metapaths that
the naive null over- or under-states because of degree confounding, and they
do — the top MC-ranked metapaths (`GpBPpGpBP`, `GpPWpGpBP`, `GpMFpGpBP`) are
also analytically high, but the middle of the ranking reorders substantially
(the broad scatter around, not tight against, the `y = x` line in the
figure).

This connects to the validation task's own history for this null design:
across its successive redesigns, the fraction of validation rows that passed
calibration fell from 85% to 47% to 14.7% as the test was made genuinely
discriminating rather than one the partition could pass vacuously, on the
way to the capacity-hurdle-adaptive (S1) design that ships here — measured
in `docs/tasks/capacity-hurdle-adaptive-null/verification.md` on branch
`fix/random-null-stratified-srswor` (see Conclusions). A weak rank
correlation here is consistent with that history: the null this task ships
is the one that survived a genuinely discriminating test, not the one that
agreed with everything it was compared against.


## Behaviour changes

Two changes a reviewer must accept as the cost of this swap, verbatim in
substance from the design (`design.md`, "Behaviour changes the summary must
present"):

**1. Deterministic.** Identical queries return identical z. `b` and `seed`
are inert — they are still accepted as parameters (so no caller breaks), but
ignored, and passing either emits a `DeprecationWarning`. Two calls to
`query_metapath_z` with the same gene set and target return byte-identical
frames (`verification.md`, "Determinism (direct function, run1 vs run2
byte-identical frames): True").

**2. Cost no longer buys noise.** Measured on the example query (verify
step, 2026-09-01): end-to-end latency is disk-dominated and roughly
unchanged (new 103 s vs old 85 s over 52 metapaths), and the warm-matrix
null step is slower per metapath (70 ms vs 8.6 ms) — at the app's default
`b = 20` the old null was fast *because* it was imprecise: a null standard
deviation estimated from 20 draws carries roughly 16% relative error, which
every z-score inherits. The analytical moments price out at the
Monte-Carlo size needed for comparable precision (validated 214x / 2,145x
against `B = 1,000` / `B = 10,000`), and the `b` knob is gone entirely. The
design's original order-of-magnitude latency expectation was written
against the wrong yardstick and is corrected here (`decisions.md`,
2026-09-01).


## Conclusions and the reviewer's onward path

- The null is deterministic and no longer has a tunable sample size; the
  cost of computing it did not improve at the app's actual `b = 20` default,
  and the design's speed claim is corrected to the measured, disk-dominated
  reality rather than left standing.
- The rank reordering relative to the old null is real, weak, and honestly
  reported — consistent with the validation history behind this design, not
  evidence of a bug.
- [`audit.md`](audit.md) — the design read as 42 numbered claims against the
  landed tree, forward and reverse: **passes**. 18 documentary findings
  (F1-F18), 17 fixed and 1 recorded in `decisions.md` (the verify-step
  controls ran via direct `query_metapath_z` calls rather than through the
  live Streamlit app, for a documented memory reason); none required a code
  change.
- [`verification.md`](verification.md) — every command run and its real
  output, including the full `pytest -q` suite pass (65 passed, 4 subtests
  passed) and the memory-bounded methodology note that explains the
  end-to-end timing figure's disk-I/O dominance.
- The null's evidence base:
  [`docs/tasks/capacity-hurdle-adaptive-null/`](https://github.com/lagillenwater/multi-dwpc/tree/fix/random-null-stratified-srswor/docs/tasks/capacity-hurdle-adaptive-null)
  on branch `fix/random-null-stratified-srswor` (commit `3b0bdee`) — the
  calibration and pass-rate-history evidence `analytical_gene_set_z` is
  built on.

This is gate 2. Human review happens here, before the pull request opens.
